# Windowing + PCA — Pipeline v2 (WINDOW_SIZE=60)

**What changed from `module3_pipeline/windowing_pca.ipynb`**: only
`WINDOW_SIZE` (30 -> 60). Everything else — gap-aware construction, PCA at
99% variance + whitening, fit on `cc1_train` only — is identical methodology.

**Why 60**: a systematic ablation (`experiments/window_size_ablation.ipynb`,
`window_size_45_ablation.ipynb`, `window_size_50_ablation.ipynb`) tested
30/45/50/60 timesteps, holding the model architecture fixed. Window=60 gave
by far the largest improvement on `cc1_test` (F1 0.618 -> 0.912) of any size
tested, and — after fixing a threshold-calibration bug specific to larger
windows (see `adaptive_threshold_blended.ipynb` in this v2 pipeline) — the
adaptive threshold and incremental learning mechanisms work correctly with
it, same as they do at window=30.

**Known trade-off, carried forward honestly**: window=60 is substantially
weaker on `drift_cc2` than window=30 (a real, structural cost — see
`final_comparison.ipynb` in this folder for the full picture). This pipeline
version is being adopted primarily for its `cc1_test` gains; the drift-side
trade-off is not hidden.

Input: `data/processed/{cc1_train,cc1_val,cc1_test,drift_complex_case2}.csv`
(from `clean_and_split.ipynb` — unchanged, reused as-is from `module3_pipeline/`,
since window size does not affect data cleaning/splitting).

In [1]:
import pandas as pd
import numpy as np
import joblib, os
from sklearn.decomposition import PCA

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
OUT_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
MODEL_DIR = os.path.join(BASE, 'models_v2')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]

SPLIT_FILES = {
    'cc1_train':      'cc1_train.csv',
    'cc1_val':        'cc1_val.csv',
    'cc1_test':       'cc1_test.csv',
    'drift_cc2':      'drift_complex_case2.csv',
}
# Only CC2 is carried forward as the drift-evaluation set (SC1/SC2 out of scope,
# same decision as the v1 pipeline's vae_eval.ipynb).

WINDOW_SIZE   = 60     # v2: up from 30 in the original pipeline
STRIDE        = 1
PCA_VARIANCE  = 0.99

print('Paths and constants configured.')
print(f'WINDOW_SIZE={WINDOW_SIZE}  STRIDE={STRIDE}  PCA_VARIANCE={PCA_VARIANCE}')

Paths and constants configured.
WINDOW_SIZE=60  STRIDE=1  PCA_VARIANCE=0.99


## Step 1 — Load split files

In [2]:
splits = {}
for name, fname in SPLIT_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    d = pd.read_csv(path, low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    splits[name] = d
    print(f'  {name:10s}: {len(d):>7,} rows  |  containers: {d["cmdb_id"].nunique()}  |  '
          f'anomalies: {int(d["label"].sum()):,}')

  cc1_train : 156,479 rows  |  containers: 27  |  anomalies: 0
  cc1_val   :  22,356 rows  |  containers: 27  |  anomalies: 0
  cc1_test  :  44,968 rows  |  containers: 27  |  anomalies: 256
  drift_cc2 :  77,760 rows  |  containers: 27  |  anomalies: 372


## Step 2 — Sliding-window construction (gap-aware, identical logic to v1)

In [3]:
def build_windows(container_df, feature_cols, window_size, stride):
    data    = container_df[feature_cols].values.astype(np.float32)
    labels  = container_df['label'].values
    ftypes  = container_df['failure_type'].values.astype(object)
    is_gap  = container_df['is_gap'].values
    n       = len(data)

    X, y, ft = [], [], []
    for i in range(0, n - window_size + 1, stride):
        if is_gap[i : i + window_size].any():
            continue
        X.append(data[i : i + window_size])
        window_labels = labels[i : i + window_size]
        y.append(int(window_labels.any()))
        w_types = sorted({t for t in ftypes[i : i + window_size] if isinstance(t, str)})
        ft.append(','.join(w_types) if w_types else None)

    if not X:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32),
                np.empty((0,), dtype=np.int64), np.array([], dtype=object))
    return np.stack(X), np.array(y, dtype=np.int64), np.array(ft, dtype=object)


def window_split(df, feature_cols, window_size, stride):
    Xs, ys, fts = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        X, y, ft = build_windows(g, feature_cols, window_size, stride)
        if len(X):
            Xs.append(X); ys.append(y); fts.append(ft)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(fts)

print('Window builder defined.')

Window builder defined.


## Step 3 — Apply windowing to every split

In [4]:
windowed = {}
for name, d in splits.items():
    X, y, ft = window_split(d, FEATURE_COLS, WINDOW_SIZE, STRIDE)
    windowed[name] = {'X': X, 'y': y, 'ft': ft}
    n_anom = int(y.sum())
    print(f'  {name:10s}: {len(d):>7,} rows -> {len(y):>7,} windows  '
          f'({n_anom:,} anomaly windows, {n_anom / max(len(y),1) * 100:.2f}%)')

  cc1_train : 156,479 rows -> 152,456 windows  (0 anomaly windows, 0.00%)
  cc1_val   :  22,356 rows ->  20,763 windows  (0 anomaly windows, 0.00%)
  cc1_test  :  44,968 rows ->  43,375 windows  (256 anomaly windows, 0.59%)
  drift_cc2 :  77,760 rows ->  76,167 windows  (1,080 anomaly windows, 1.42%)


## Step 4 — Flatten windows

In [5]:
for name in windowed:
    X = windowed[name]['X']
    X_flat = X.reshape(len(X), -1)
    windowed[name]['X_flat'] = X_flat
    print(f'  {name:10s}: {X.shape} -> {X_flat.shape}')

  cc1_train : (152456, 60, 7) -> (152456, 420)
  cc1_val   : (20763, 60, 7) -> (20763, 420)
  cc1_test  : (43375, 60, 7) -> (43375, 420)
  drift_cc2 : (76167, 60, 7) -> (76167, 420)


## Step 5 — PCA (fit on `cc1_train` only, 99% variance + whitening)

In [6]:
X_train_flat = windowed['cc1_train']['X_flat']
print(f'Fitting PCA on {len(X_train_flat):,} cc1_train windows (all normal) ...')
print(f'Input dimension: {X_train_flat.shape[1]} (= {WINDOW_SIZE} timesteps x {len(FEATURE_COLS)} features)')

pca = PCA(n_components=PCA_VARIANCE, svd_solver='full', whiten=True, random_state=42)
pca.fit(X_train_flat)
n_components = pca.n_components_
print(f'Components retained for {PCA_VARIANCE*100:.0f}% variance: {n_components}  '
      f'(compression {X_train_flat.shape[1]} -> {n_components})')
print(f'(v1 pipeline, window=30: 210 -> 26 components, for reference)')

for name in windowed:
    windowed[name]['X_pca'] = pca.transform(windowed[name]['X_flat']).astype(np.float32)
    print(f'  {name:10s}: {windowed[name]["X_flat"].shape} -> {windowed[name]["X_pca"].shape}')

Fitting PCA on 152,456 cc1_train windows (all normal) ...
Input dimension: 420 (= 60 timesteps x 7 features)
Components retained for 99% variance: 48  (compression 420 -> 48)
(v1 pipeline, window=30: 210 -> 26 components, for reference)
  cc1_train : (152456, 420) -> (152456, 48)
  cc1_val   : (20763, 420) -> (20763, 48)
  cc1_test  : (43375, 420) -> (43375, 48)
  drift_cc2 : (76167, 420) -> (76167, 48)


## Step 6 — Save windows + PCA model

In [7]:
for name, d in windowed.items():
    np.save(os.path.join(OUT_DIR, f'X_{name}.npy'),  d['X_pca'])
    np.save(os.path.join(OUT_DIR, f'y_{name}.npy'),  d['y'])
    np.save(os.path.join(OUT_DIR, f'ft_{name}.npy'), d['ft'])
    print(f'  saved X/y/ft_{name}.npy  ({len(d["y"]):,} windows)')

pca_path = os.path.join(MODEL_DIR, 'cc1_pca.pkl')
joblib.dump({
    'pca': pca,
    'feature_cols': FEATURE_COLS,
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'n_components': n_components,
}, pca_path)
print(f'\n  PCA model saved -> {pca_path}')

  saved X/y/ft_cc1_train.npy  (152,456 windows)
  saved X/y/ft_cc1_val.npy  (20,763 windows)
  saved X/y/ft_cc1_test.npy  (43,375 windows)
  saved X/y/ft_drift_cc2.npy  (76,167 windows)

  PCA model saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models_v2\cc1_pca.pkl


## Step 7 — Final verification

In [8]:
print('=== FINAL VERIFICATION ===\n')
for name in windowed:
    X = np.load(os.path.join(OUT_DIR, f'X_{name}.npy'))
    y = np.load(os.path.join(OUT_DIR, f'y_{name}.npy'))
    print(f'{name:10s} X={str(X.shape):>16s}  y={str(y.shape):>10s}  '
          f'anomalies={int(y.sum()):>5,}  NaNs={int(np.isnan(X).sum())}  '
          f'range=[{X.min():.3f}, {X.max():.3f}]')

=== FINAL VERIFICATION ===

cc1_train  X=    (152456, 48)  y= (152456,)  anomalies=    0  NaNs=0  range=[-21.426, 27.836]
cc1_val    X=     (20763, 48)  y=  (20763,)  anomalies=    0  NaNs=0  range=[-11.671, 16.106]
cc1_test   X=     (43375, 48)  y=  (43375,)  anomalies=  256  NaNs=0  range=[-26.473, 39.280]
drift_cc2  X=     (76167, 48)  y=  (76167,)  anomalies=1,080  NaNs=0  range=[-36.976, 32.089]


## Output files

| File | Shape | Purpose |
|---|---|---|
| `data/processed/windows_cc1_v2/X_cc1_train.npy` / `y_...` / `ft_...` | (N, n_pc) | VAE training input — all normal |
| `X_cc1_val.npy` | (N, n_pc) | early stopping (all normal) |
| `X_cc1_test.npy` | (N, n_pc) | in-distribution eval |
| `X_drift_cc2.npy` | (N, n_pc) | drift-evaluation set |
| `models_v2/cc1_pca.pkl` | -- | fitted PCA + window config (WINDOW_SIZE=60) |

**Next:** `train_vae.ipynb` (v2) — loads these arrays, trains on
`X_cc1_train.npy`, same hyperparameters as the validated v1 model.